In [3]:
#https://github.com/lucidrains/CoCa-pytorch?tab=readme-ov-file
!pip install vit-pytorch

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple


In [1]:
import torch

# import vision transformer

from vit_pytorch.simple_vit_with_patch_dropout import SimpleViT
from vit_pytorch.extractor import Extractor

vit = SimpleViT(
    image_size = 256,
    patch_size = 32,
    num_classes = 1000,
    dim = 1024,
    depth = 6,
    heads = 16,
    mlp_dim = 2048,
    patch_dropout = 0.5  # https://arxiv.org/abs/2212.00794
)

vit = Extractor(vit, return_embeddings_only = True, detach = False)

# extractor will enable it so the vision transformer returns its embeddings

# import CoCa and instantiate it

from coca_pytorch.coca_pytorch import CoCa

coca = CoCa(
    dim = 512,                     # model dimension
    img_encoder = vit,             # vision transformer - image encoder, returning image embeddings as (batch, seq, dim)
    image_dim = 1024,              # image embedding dimension, if not the same as model dimensions
    num_tokens = 20000,            # number of text tokens
    unimodal_depth = 6,            # depth of the unimodal transformer
    multimodal_depth = 6,          # depth of the multimodal transformer
    dim_head = 64,                 # dimension per attention head
    heads = 8,                     # number of attention heads
    caption_loss_weight = 1.,      # weight on the autoregressive caption loss
    contrastive_loss_weight = 1.,  # weight on the contrastive loss between image and text CLS embeddings
).cuda()

# mock text and images

text = torch.randint(0, 20000, (4, 512)).cuda()
images = torch.randn(4, 3, 256, 256).cuda()

# train by giving CoCa your text and images with `return_loss = True`

loss = coca(
    text = text,
    images = images,
    return_loss = True  # set this to True to get the full caption + contrastive loss
)

loss.backward()

# do the above for as much text and images...
# then you can get the caption logits as so

logits = coca(
    text = text,
    images = images
) # (4, 512, 20000)

# and the CLIP-like text and image embeddings as

text_embeds, image_embeds = coca(
    text = text,
    images = images,
    return_embeddings = True
) # (4, 512), (4, 512)

print(text_embeds) # (4, 512), (4, 512)

tensor([[ 0.2377, -0.4746, -0.2257,  ..., -0.6425, -2.0029,  0.4063],
        [-0.0093, -1.5064, -0.1115,  ..., -0.9403, -0.5591, -0.6960],
        [-0.4304, -0.7972, -0.8556,  ..., -1.1375, -0.6704,  0.2818],
        [-0.3968, -1.1182,  0.1739,  ..., -1.0501, -0.9566, -0.1180]],
       device='cuda:0', grad_fn=<NativeLayerNormBackward0>)


In [1]:
import torch, torch.nn as nn
from torchvision import transforms
from PIL import Image
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from vit_pytorch.simple_vit_with_patch_dropout import SimpleViT
from vit_pytorch.extractor import Extractor
from coca_pytorch.coca_pytorch import CoCa

dtype = torch.float16
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-8B")
qwen = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-8B", torch_dtype=torch.float16, device_map="auto").eval()
hidden_size = qwen.config.hidden_size

vit = SimpleViT(
    image_size=224, patch_size=32, num_classes=1000,
    dim=768, depth=4, heads=8, mlp_dim=1024, patch_dropout=0.3
)
vit = Extractor(vit, return_embeddings_only=True, detach=False)

coca = CoCa(
    dim=384,
    img_encoder=vit,
    image_dim=768,
    num_tokens=20000,
    unimodal_depth=2,
    multimodal_depth=2,
    dim_head=64,
    heads=4,
    caption_loss_weight=1.0,
    contrastive_loss_weight=1.0,
).to(device).eval()

proj = nn.Linear(384, hidden_size, bias=False).to(device)

prep = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

ds = load_dataset("zmao/food_img_caption_small", split="train")

def collate_fn(batch):
    imgs = torch.stack([prep(x["image"]) for x in batch]).to(device)
    caps = [x["text"] + tokenizer.eos_token for x in batch]
    cap_ids = tokenizer(caps, return_tensors="pt", padding=True).input_ids.to(device)
    return imgs, cap_ids

loader = torch.utils.data.DataLoader(ds, batch_size=2, shuffle=True, collate_fn=collate_fn)
opt = torch.optim.AdamW(proj.parameters(), lr=2e-4)

for epoch in range(1):
    print(f"Epoch {epoch+1}")
    proj.train()
    for imgs, cap_ids in loader:
        with torch.no_grad():
            _, img_emb = coca(
                text=torch.zeros((imgs.size(0), 1), dtype=torch.long, device=device),
                images=imgs,
                return_embeddings=True
            )
        vis_tok = proj(img_emb).unsqueeze(1)
        with torch.no_grad():
            cap_emb = qwen.model.embed_tokens(cap_ids[:, :-1])
        inp_emb = torch.cat([vis_tok, cap_emb], dim=1)
        attn_msk = torch.ones(inp_emb.shape[:2], dtype=torch.long, device=device)
        labels = torch.full(inp_emb.shape[:2], -100, dtype=torch.long, device=device)
        labels[:, vis_tok.size(1):] = cap_ids[:, 1:]
        loss = qwen(inputs_embeds=inp_emb, attention_mask=attn_msk, labels=labels).loss
        loss.backward()
        print(f"Loss: {loss.item():.4f}")
        opt.step(); opt.zero_grad()

proj.eval()
first = ds[0]
img_t = prep(first["image"]).unsqueeze(0).to(device)
with torch.no_grad():
    _, img_emb = coca(
        text=torch.zeros((1, 1), dtype=torch.long, device=device),
        images=img_t,
        return_embeddings=True
    )
    vis_tok = proj(img_emb).unsqueeze(1)
    prompt = "What do you see in this image?"
    chat = tokenizer.apply_chat_template([{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True)
    ids = tokenizer(chat, return_tensors="pt").input_ids.to(device)
    txt_emb = qwen.model.embed_tokens(ids)
    inp_emb = torch.cat([vis_tok, txt_emb], dim=1)
    attn_msk = torch.ones(inp_emb.shape[:2], dtype=torch.long, device=device)
    out = qwen.generate(inputs_embeds=inp_emb, attention_mask=attn_msk, max_new_tokens=128, do_sample=False)
    gen = out[0][ids.shape[1]:]
    print(tokenizer.decode(gen, skip_special_tokens=True))


/home/cqilab/anaconda3/envs/llmfinetune/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-05-22 15:15:48.139522: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-05-22 15:15:48.153498: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1747894548.170707   68280 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1747894548.174447   68280 cuda_blas

Epoch 1


RuntimeError: expected mat1 and mat2 to have the same dtype, but got: float != c10::Half